In [1]:
import pandas as pd

train=pd.read_csv(r"C:\Users\33651\Desktop\ML_PROJECT\Heart_prediction\train.csv")
test=pd.read_csv(r"C:\Users\33651\Desktop\ML_PROJECT\Heart_prediction\test.csv")
train.head()


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [2]:
test.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


In [3]:
train["Heart Disease"].value_counts()

Heart Disease
Absence     347546
Presence    282454
Name: count, dtype: int64

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

In [5]:
train.isnull().sum()

id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
dtype: int64

In [6]:
train["Heart Disease"]=train["Heart Disease"].map({"Absence":0,"Presence":1})

train["Heart Disease"]

0         1
1         0
2         0
3         0
4         1
         ..
629995    0
629996    0
629997    1
629998    1
629999    0
Name: Heart Disease, Length: 630000, dtype: int64

In [7]:
from sklearn.model_selection import train_test_split

y=train["Heart Disease"]

X=train.drop("Heart Disease",axis=1)

X_train,X_valid,y_train,y_valid=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=42)



In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models={
    "DecisionTree": (
        DecisionTreeClassifier(random_state=42),
        {
            "model__max_depth": [None, 5, 10],
            "model__min_samples_split": [2, 5]
        }
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {
            "model__n_estimators": [50, 100, 500],
            "model__max_depth": [None, 5, 10]
        }
    ),
    "XGBoost": (
        XGBClassifier(random_state=42),
        {
            "model__n_estimators": [50, 100, 200],
            "model__learning_rate": [0.01, 0.1, 0.2],
            "model__max_depth": [3, 5, 7]
        }
    )
}

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score

results=[]
parameters=[]

for name,(model,params) in models.items():
    my_pipeline=Pipeline(steps=[("model",model)])

    grid=GridSearchCV(my_pipeline,param_grid=params,cv=5,scoring="accuracy",n_jobs=-1)

    grid.fit(X_train,y_train)

    prediction_train=grid.predict(X_valid)

    results.append({
        "Model":name,
        "Accuracy":accuracy_score(y_valid,prediction_train),
        "Confusion Matrix": confusion_matrix(y_valid,prediction_train),
        "Precision":precision_score(y_valid,prediction_train),
        "Recall": recall_score(y_valid,prediction_train),
        "F1 score": f1_score(y_valid,prediction_train)
    })

    parameters.append({
        "Model":name,   
        "best param":grid.best_params_
    })

In [10]:
results_df=pd.DataFrame(results)
results_df.sort_values(by="Accuracy",ascending=False,inplace=True)
results_df.loc[:,["Model","Accuracy"]]

,Model,Accuracy
2,XGBoost,0.887333
1,RandomForest,0.884405
0,DecisionTree,0.880913


In [11]:
results_df

,Model,Accuracy,Confusion Matrix,Precision,Recall,F1 score
2,XGBoost,0.887333,"[[62903, 6661], [7535, 48901]]",0.880116,0.866486,0.873248
1,RandomForest,0.884405,"[[62855, 6709], [7856, 48580]]",0.878656,0.860798,0.869635
0,DecisionTree,0.880913,"[[62667, 6897], [8108, 48328]]",0.875111,0.856333,0.865620


In [12]:
parameters_df=pd.json_normalize(parameters)
parameters_df.loc[2:]

,Model,best param.model__max_depth,best param.model__min_samples_split,best param.model__n_estimators,best param.model__learning_rate
2,XGBoost,3,NaN,200.0,0.2


In [13]:
X_test=test

xgb_model=XGBClassifier(max_depth=3,n_estimators=200,learning_rate=0.2)

xgb_model.fit(X_train,y_train)

prediction_test=xgb_model.predict(X_test)

prediction_test_df=pd.DataFrame({"Disease Prediction":prediction_test})

prediction_test_df.to_csv(r"C:\Users\33651\Desktop\ML_PROJECT\Heart_prediction\Heart_prediction_test.csv",index=False)



In [18]:
df=pd.read_csv(r"C:\Users\33651\Desktop\ML_PROJECT\Heart_prediction\Heart_prediction_test.csv")
df.loc[:10]

,Disease Prediction
0,1
1,0
2,1
3,0
4,0
5,1
6,0
7,1
8,1
9,0
